# À l'origine : le sac de mots

## Données

Repartons des données que nous avons mis en forme, et amenons progressivement une analyse plus avancée

In [ ]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/pyshs/URFIST-Lyon-2026/refs/heads/main/data/css_openalex_26022026.csv")
df.head()

,id,type,primary_location,title,abstract_inverted_index,publication_year,publication_date,open_access,relevance_score,abstract,journal
0,https://openalex.org/W2159397589,article,"{'id': 'doi:10.1126/science.1167742', 'is_oa':...",Computational Social Science,"{'A': [0], 'field': [1], 'is': [2], 'emerging'...",2009.0,2009-02-06,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",1360.35770,A field is emerging that leverages the capacit...,Science
1,https://openalex.org/W2070907364,article,"{'id': 'doi:10.1140/epjst/e2012-01697-8', 'is_...",Manifesto of computational social science,NaN,2012.0,2012-11-01,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...",497.82666,NaN,The European Physical Journal Special Topics
2,https://openalex.org/W3081158114,article,"{'id': 'doi:10.1126/science.aaz8170', 'is_oa':...",Computational social science: Obstacles and op...,"{'Data': [0], 'sharing,': [1], 'research': [2]...",2020.0,2020-08-28,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",438.53986,"Data sharing, research ethics, and incentives ...",Science
3,https://openalex.org/W3022499311,article,{'id': 'doi:10.1146/annurev-soc-121919-054621'...,Computational Social Science and Sociology,"{'The': [0], 'integration': [1], 'of': [2, 16,...",2020.0,2020-04-28,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...",413.03424,The integration of social science with compute...,Annual Review of Sociology
4,https://openalex.org/W3174174150,article,"{'id': 'doi:10.1038/s41586-021-03659-0', 'is_o...",Integrating explanation and prediction in comp...,NaN,2021.0,2021-06-30,"{'is_oa': False, 'oa_status': 'closed', 'oa_ur...",408.08980,NaN,Nature


## Transformer le texte en nombres : le comptage de mots

### Découper en mots

In [ ]:
df["texte"].apply(lambda x: x.split(" ")).head()

### Utiliser NLTK

In [ ]:
#pip install nltk

In [1]:
import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize
texte = "Ceci est un test. Coucou, je fais du Python. N'est-ce-pas?"
word_tokenize(texte, language="french")

[nltk_data] Downloading package punkt to /Users/emilien/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


['Ceci',
 'est',
 'un',
 'test',
 '.',
 'Coucou',
 ',',
 'je',
 'fais',
 'du',
 'Python',
 '.',
 "N'est-ce-pas",
 '?']

Les limites de NLTK (notamment le français)

#### De nombreuses fonctions 

Compter les bigrams

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
text = "Ceci est un test. Coucou, je fais du Python. N'est-ce-pas?"
tokens = word_tokenize(text.lower())
bigrams = list(ngrams(tokens, 2))

### Préprocessing : nettoyer les données

- Beaucoup d'opérations de nettoyage, spécifiques à chaque tache

In [ ]:
nltk.download("stopwords")

from nltk.corpus import stopwords

english_stopwords = list(set(stopwords.words("english")))


def generate_bigrams_nltk(text):
    tokens = word_tokenize(text.lower())
    filtered_tokens = [token for token in tokens if token.isalnum() and token not in english_stopwords]
    bigrams = list(ngrams(filtered_tokens, 2))
    return bigrams

generate_bigrams_nltk(df.loc[0, "texte"])


## DTM : document term matrix

Le faire à la main. Ou rapide avec Scikit-learn

In [ ]:
#pip install scikit-learn

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# créer mon object de ML
vectorizer = CountVectorizer(stop_words=english_stopwords, 
                             ngram_range=(1, 1), 
                             max_features=800)

# appliquer sur les données
X = vectorizer.fit_transform(df["texte"])
X = pd.DataFrame(X.toarray(),columns=list(vectorizer.get_feature_names_out()))
X

## Une version un peu plus avancée

- Term Frequency-Inverse Document Frequency
    - Amélioration du DTM
- Approche souvent utilisée pour mettre en valeur les mots les plus spécifiques
- `Scikit-learn` a `TfidfVectorizer`

$$\text{TF-IDF}(t, d, D) = \left( \frac{f_{t,d}}{n_d} \right) \times \log \left(\frac{N}{\text{df}_t} \right)
$$

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords

stopwords_english = list(set(stopwords.words("english")))

# créer un objet
vectorizer = TfidfVectorizer(stop_words=stopwords_english, 
                             ngram_range=(1, 1), 
                             max_features=500)

# applique 
X = vectorizer.fit_transform(df["texte"])

# mettre en forme
X = pd.DataFrame(X.toarray(),columns=list(vectorizer.get_feature_names_out()))
X

Les mots les plus significatifs

In [ ]:
X.loc[120].sort_values()

Distance entre deux éléments

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_similarity(X.loc[120, :].values.reshape(1, -1), 
                  X.loc[124, :].values.reshape(1, -1))

Toutes les distances, et récupérer le document le plus proche

In [ ]:
from sklearn.metrics.pairwise import pairwise_distances

distances = pd.DataFrame(pairwise_distances(X, metric="cosine"))
distances[199].idxmax()

### Application avec le clustering

Créer la matrice de distances

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt

Z = linkage(X.values, method='ward') 

Représenter la figure

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 5))
dendrogram(Z, orientation='right')
plt.title("Classification Hiérarchique Ascendante (CHA)")
plt.tight_layout()
plt.show()

Clustering

In [ ]:
from sklearn.cluster import AgglomerativeClustering
cluster = AgglomerativeClustering(n_clusters=5,  linkage='ward')
labels = cluster.fit_predict(X.values)
df['cluster'] = labels

Contenu des clusters

In [ ]:
df_w = pd.DataFrame(X.values, 
                    columns=vectorizer.get_feature_names_out()) 
df_w[df["cluster"] == 2].max().sort_values()[-20:]

## Application : Faire un nuage de mots avec WordCloud

Un coup d'oeil à la [documentation](https://amueller.github.io/word_cloud/)